### **Script for extracting morphometry measures (anatomical region volumes) from FreeSurfer subject data.**

- Assumes default locations for FreeSurfer subject data directories
- Harvests measures from multiple different '.stats' files for each subject, and combines them
- Exports results as one or multiple flattened dataframe/csv file(s), with one row per unique subject

**Notes on 'GLOBAL__...' feature columns:**

These are whole-brain measures of various kinds, and include:

- **eTIV:** *Estimated Total Intracranial Volume*, a FreeSurfer-derived estimate of total intracranial space used as a proxy for head size; also used for eTIV-based normalization (highly recommended). Primary nuisance covariate capturing head size, and commonly used to normalize or control for global scaling effects across subjects.

- **BrainSegVolNotVent:** Total brain tissue volume excluding ventricular spaces, representing overall brain parenchymal volume. Useful as a normalization denominator and as a covariate for overall brain size independent of ventricular enlargement.

- **CortexVol:** Total cortical gray matter volume summed across both hemispheres; can act as a coarse marker of cortical atrophy or developmental differences.

- **CerebralWhiteMatterVol:** Total cerebral white matter volume across both hemispheres; can capture large-scale white matter differences that may confound or contextualize regional measures.

- **SubCortGrayVol:** Total volume of subcortical gray matter structures (e.g., basal ganglia, thalamus); often informative for disease-related patterns, and sometimes useful as a low-dimensional structural feature.

- **TotalGrayVol:** Total gray matter volume, including both cortical and subcortical gray matter; captures overall gray matter burden and useful for modeling global neuroanatomical variation.

- **VentricleChoroidVol:** Combined volume of the ventricular system and choroid plexus; marker often associated with neurodegeneration, aging, or other large-scale brain-size effects.


--------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os
import re
import datetime
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path
from typing import List, Optional, Dict, Any
from collections import Counter
from datetime import datetime

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = Path(config['freesurfer']['home'])
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = Path(config['freesurfer']['subjects_dir'])
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
### SET PARAMETERS:
HARD_STOP = config['hard_errors']
SUBSET = config['subset']

MORPHOMETRY_OUTPUT_DIR = config['morphometry']['morphometry_output_dir']

MORPHOMETRY_COLUMN_PREFIX = config['morphometry']['morphometry_feature_prefix']
CALCULATE_BILATERAL_FEATURES = config['morphometry']['calculate_bilateral_features']
STANDARDIZE_COLUMN_NAMES = config['morphometry']['standardize_morphometry_feature_columns']

EXPORT_REPRESENTATIONS = config['morphometry']['export_representations']

# EXPORT_MODE = config['morphometry']['export_mode']    # TEMPORARILY FROZEN: It complicates downstream wrangling too much. 'Single' mode already labels columns by thematic groups, so downstream selection by users should be straightforward enough
EXPORT_MODE = 'single'

### SET PATHS:
ROOT_DIR = Path(config['root_output_directory'])

# Output:
OUTPUT_PATH = Path(ROOT_DIR) / MORPHOMETRY_OUTPUT_DIR
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

### LOAD MANIFESTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)

MEG_MANIFEST_PATH = Path(ROOT_DIR) / 'MEG_manifest.csv' 
MEG_runs = pd.read_csv(MEG_MANIFEST_PATH)

In [ ]:
### DIAGNOSTIC SUBSETTING (if enabled):
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Selecting all subject_IDs from first {SUBSET} available MEG files:")
    subset_subjects = (
        MEG_runs
        .head(SUBSET)
        .loc[:, "subject_ID"]
        .dropna()
        .astype(str)
        .unique())
    before_n = len(RUN_MANIFEST)
    RUN_MANIFEST = (
        RUN_MANIFEST[RUN_MANIFEST["subject_ID"].astype(str).isin(subset_subjects)].copy())
    after_n = len(RUN_MANIFEST)
    print(f"  → Subsetting: {before_n} → {after_n}")
    print(f"  → MRI volume reconstructions will be performed for {after_n} subject_IDs.")
    display(RUN_MANIFEST)

In [ ]:
# ---- Extract unique subject IDs from manifest
if "subject_ID" not in MEG_runs.columns:
    raise KeyError("Expected column 'subject_ID' not found in MEG_runs.")

subject_list = (
    MEG_runs["subject_ID"].dropna().astype(str).unique().tolist())

subject_list = sorted(subject_list)

if len(subject_list) == 0:
    raise RuntimeError("No subject_ID values found in MEG_runs['subject_ID'].")

print(f"[INFO] Found {len(subject_list)} unique subject_IDs in MEG_runs.")

# ---- Validate expected FreeSurfer folder structure + key stats files
required_stats_files = ["aseg.stats", "wmparc.stats", "lh.aparc.stats", "rh.aparc.stats"]

missing_subject_dirs = []
missing_stats_dirs = []
missing_stats_files = []  # list of tuples: (subject_ID, missing_files)

for sid in subject_list:
    subj_dir = FREESURFER_SUBJECTS_DIR / sid
    stats_dir = subj_dir / "stats"

    if not subj_dir.is_dir():
        missing_subject_dirs.append(sid)
        continue

    if not stats_dir.is_dir():
        missing_stats_dirs.append(sid)
        continue

    missing_files_this_subj = [f for f in required_stats_files if not (stats_dir / f).is_file()]
    if missing_files_this_subj:
        missing_stats_files.append((sid, missing_files_this_subj))

# ---- Report validation results
n_total = len(subject_list)

if missing_subject_dirs:
    print(f"\n[WARN] {len(missing_subject_dirs)}/{n_total} subjects missing FreeSurfer subject directory:")
    print("       " + ", ".join(missing_subject_dirs[:20]) + (" ..." if len(missing_subject_dirs) > 20 else ""))

if missing_stats_dirs:
    print(f"\n[WARN] {len(missing_stats_dirs)}/{n_total} subjects missing 'stats/' directory:")
    print("       " + ", ".join(missing_stats_dirs[:20]) + (" ..." if len(missing_stats_dirs) > 20 else ""))

if missing_stats_files:
    print(f"\n[WARN] {len(missing_stats_files)}/{n_total} subjects missing one or more required *.stats files.")
    # Show a small sample for readability
    for sid, mf in missing_stats_files[:10]:
        print(f"       - {sid}: missing {mf}")
    if len(missing_stats_files) > 10:
        print("       ...")

# ---- Decide whether to stop or filter subjects down to valid set
invalid_subjects = set(missing_subject_dirs) | set(missing_stats_dirs) | set([sid for sid, _ in missing_stats_files])

valid_subject_list = [sid for sid in subject_list if sid not in invalid_subjects]

print(f"\n[INFO] Valid FreeSurfer subjects with required stats files: {len(valid_subject_list)}/{n_total}")

if len(valid_subject_list) == 0:
    raise RuntimeError("After validation, zero subjects have the required FreeSurfer stats files.")

if invalid_subjects and HARD_STOP:
    raise RuntimeError(
        f"HARD_STOP is enabled and {len(invalid_subjects)} subject(s) are missing required FreeSurfer inputs. "
        "Inspect warnings above.")

# Final list to use downstream:
subject_list = valid_subject_list

In [ ]:
# __________________________________________________________________________________________________________
### FUNCTION: Extract table data from a FreeSurfer *.stats file

def extract_brain_volume_stats(
    subject_id: str,
    stats_stem: str,
    subjects_dir: Path
) -> Optional[pd.DataFrame]:
    """
    Extract the table data from a FreeSurfer .stats file for a given subject and return it as a DataFrame.
    """

    subjects_dir = Path(subjects_dir)
    stats_dir = subjects_dir / str(subject_id) / "stats"
    target_file = stats_dir / f"{stats_stem}.stats"

    column_names = None
    data = []

    try:
        if not target_file.is_file():
            raise FileNotFoundError(f"Stats file not found: {target_file}")

        lines = target_file.read_text(errors="replace").splitlines()

        for line in reversed(lines):
            if line.startswith("#"):
                possible_header = line.lstrip("#").strip()
                if possible_header.startswith("ColHeaders"):
                    column_names = possible_header[len("ColHeaders"):].strip().split()
                    break

        if column_names is None:
            raise ValueError("Column headers not found (no '# ColHeaders' line detected).")

        for line in lines:
            if not line or line.startswith("#"):
                continue
            row = re.split(r"\s+", line.strip())
            if row:
                data.append(row)

        if not data:
            raise ValueError("No data rows parsed from stats file.")

        max_columns = max(len(row) for row in data)

        if len(column_names) != max_columns:
            column_names = [f"Column_{i+1}" for i in range(max_columns)]

        padded_data = [row + [""] * (max_columns - len(row)) for row in data]

        df = pd.DataFrame(padded_data, columns=column_names).reset_index(drop=True)
        return df

    except Exception as e:
        print(f"[WARN] Failed to process {stats_stem}.stats for subject '{subject_id}': {e}")
        return None


# __________________________________________________________________________________________________________
### FUNCTION: Extract global (header) measures from a FreeSurfer *.stats file (e.g., aseg.stats)

def extract_freesurfer_global_measures(
    subject_id: str,
    stats_stem: str,
    subjects_dir: Path,
    debug_print_first_lines: int = 0
) -> Optional[Dict[str, float]]:
    """
    Extract FreeSurfer header/global metrics from <subjects_dir>/<subject_id>/stats/<stats_stem>.stats.

    Robust to common FreeSurfer mri_segstats header formats, including 5-field Measure lines like:
      # Measure BrainSeg, BrainSegVol, Brain Segmentation Volume, 1195064.000000, mm^3
      # Measure BrainSegNotVent, BrainSegVolNotVent, ..., 1182198.000000, mm^3
      # Measure EstimatedTotalIntraCranialVol, eTIV, ..., 1556275.449763, mm^3

    Strategy:
      - For comma-delimited Measure lines:
          * split by commas
          * find the last numeric token as the value
          * store under token[0] and token[1] (if present) to maximize compatibility
      - For space-delimited Measure lines:
          * take first token as key
          * take first numeric token as value

    Returns:
        dict mapping measure_name -> float_value
        or None if none were parsed
    """
    subjects_dir = Path(subjects_dir)
    stats_path = subjects_dir / str(subject_id) / "stats" / f"{stats_stem}.stats"

    if not stats_path.exists():
        print(f"[WARN] Missing stats file for global measures: {stats_path}")
        return None

    try:
        lines = stats_path.read_text(errors="replace").splitlines()
    except Exception as e:
        print(f"[WARN] Failed reading stats file: {stats_path} ({e})")
        return None

    if debug_print_first_lines and debug_print_first_lines > 0:
        print(f"\n[DEBUG] First {debug_print_first_lines} lines of {stats_path}:")
        for ln in lines[:debug_print_first_lines]:
            print(ln)

    measures: Dict[str, float] = {}

    measure_line_regex = re.compile(r"^#\s*Measure\s+(.*)$")
    number_regex = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

    def _try_float(x: str) -> Optional[float]:
        try:
            return float(x)
        except Exception:
            return None

    for line in lines:
        if not line.startswith("#"):
            continue

        m = measure_line_regex.match(line.strip())
        if not m:
            continue

        payload = m.group(1).strip()

        # Case 1: comma-delimited (common in your file)
        if "," in payload:
            parts = [p.strip() for p in payload.split(",")]
            # Find the last numeric-like token among parts
            value = None
            for p in reversed(parts):
                v = _try_float(p)
                if v is not None:
                    value = v
                    break
            if value is None:
                # No numeric part found; skip
                continue

            # Store under the first and second fields if present
            # Example: BrainSeg, BrainSegVol, <desc>, <value>, <unit>
            if len(parts) >= 1 and parts[0]:
                measures[parts[0]] = value
            if len(parts) >= 2 and parts[1]:
                measures[parts[1]] = value
            continue

        # Case 2: space-delimited fallback
        # Example: "EstimatedTotalIntraCranialVol 1556275.449763 mm^3"
        tokens = payload.split()
        if not tokens:
            continue

        key = tokens[0].strip()
        num_match = number_regex.search(payload)
        if num_match:
            v = _try_float(num_match.group(0))
            if v is not None:
                measures[key] = v

    return measures if measures else None

In [ ]:
# __________________________________________________________________________________________________________
### MAIN EXTRACTION LOOP: build per-subject morphometry feature table (no export yet)

required_stats_stems = ["aseg", "wmparc", "lh.aparc", "rh.aparc"]

master_df_list = []
failed_subjects = []   # subjects that could not be processed (partial or full)
skipped_subjects = []  # subjects skipped due to missing critical inputs (non-hard-stop mode)
missing_global_measures_subjects = []  # subjects with no parsed '# Measure' globals

for sid in subject_list:

    # ---- Load stats tables
    stats_dfs = {}
    missing_or_failed = []

    for stem in required_stats_stems:
        df = extract_brain_volume_stats(subject_id=sid, stats_stem=stem, subjects_dir=FREESURFER_SUBJECTS_DIR)
        if df is None:
            missing_or_failed.append(stem)
        stats_dfs[stem] = df

    # ---- Handle missing inputs
    if missing_or_failed:
        msg = f"[WARN] Subject '{sid}': failed to load stats file(s): {missing_or_failed}"
        if HARD_STOP:
            raise RuntimeError(msg)
        else:
            print(msg)
            skipped_subjects.append(sid)
            continue

    aseg_df     = stats_dfs["aseg"]
    wmparc_df   = stats_dfs["wmparc"]
    lh_aparc_df = stats_dfs["lh.aparc"].copy()
    rh_aparc_df = stats_dfs["rh.aparc"].copy()

    # ---- Basic schema checks (fail fast or skip)
    # Whole-brain tables: need StructName + Volume_mm3
    wholebrain_columns_to_keep = ["StructName", "Volume_mm3"]
    hemi_columns_to_keep = ["StructName", "GrayVol"]

    missing_cols = []
    for df_name, df_obj, cols in [
        ("aseg", aseg_df, wholebrain_columns_to_keep),
        ("wmparc", wmparc_df, wholebrain_columns_to_keep),
        ("lh.aparc", lh_aparc_df, hemi_columns_to_keep),
        ("rh.aparc", rh_aparc_df, hemi_columns_to_keep)]:
        for c in cols:
            if c not in df_obj.columns:
                missing_cols.append((df_name, c))

    if missing_cols:
        msg = f"[WARN] Subject '{sid}': missing expected column(s): {missing_cols}"
        if HARD_STOP:
            raise RuntimeError(msg)
        else:
            print(msg)
            failed_subjects.append(sid)
            continue

    # ---- Prefix cortical structure names to avoid collisions and make hemisphere explicit
    # (matches your legacy behavior)
    lh_aparc_df["StructName"] = "gm_lh_" + lh_aparc_df["StructName"].astype(str)
    rh_aparc_df["StructName"] = "gm_rh_" + rh_aparc_df["StructName"].astype(str)

    # ---- Combine wholebrain structures
    wholebrain_structs = pd.concat(
        [aseg_df[wholebrain_columns_to_keep], wmparc_df[wholebrain_columns_to_keep]],
        ignore_index=True)

    # ---- Combine hemisphere structures and harmonize volume column name
    hemisphere_structs = pd.concat(
        [lh_aparc_df[hemi_columns_to_keep], rh_aparc_df[hemi_columns_to_keep]],
        ignore_index=True).rename(columns={"GrayVol": "Volume_mm3"})

    # ---- Combine all structures into one long-form table
    all_structs = pd.concat([wholebrain_structs, hemisphere_structs], ignore_index=True)

    # ---- Coerce volume to numeric where possible (keeps NaN if coercion fails)
    all_structs["Volume_mm3"] = pd.to_numeric(all_structs["Volume_mm3"], errors="coerce")

    # ---- Remove rows without a valid StructName
    all_structs["StructName"] = all_structs["StructName"].astype(str)
    all_structs = all_structs[all_structs["StructName"].str.len() > 0].copy()

    # ---- Pivot to wide (one row per subject)
    # If duplicates exist (same StructName appears multiple times), we must resolve before pivot.
    dup_mask = all_structs.duplicated(subset=["StructName"], keep=False)
    if dup_mask.any():
        dup_names = sorted(all_structs.loc[dup_mask, "StructName"].unique().tolist())
        msg = f"[WARN] Subject '{sid}': duplicate StructName entries detected (n={len(dup_names)}). Example: {dup_names[:10]}"
        if HARD_STOP:
            raise RuntimeError(msg)
        else:
            # Conservative: aggregate duplicates by sum (volumes). This is defensible for volume metrics.
            print(msg + "  --> Aggregating duplicates by sum.")
            all_structs = (
                all_structs
                .groupby("StructName", as_index=False)["Volume_mm3"]
                .sum(min_count=1))

    df_pivot = all_structs.set_index("StructName").T
    df_pivot = df_pivot.rename_axis(None, axis=1).reset_index(drop=True)

    # ---- Add identifiers (keep legacy columns for compatibility with downstream joins)
    df_pivot.insert(0, "subject_ID", sid)

    # ---- Add global aseg header measures (eTIV, BrainSegVolNotVent, etc.) from '# Measure' lines
    global_measures = extract_freesurfer_global_measures(
        subject_id=sid,
        stats_stem="aseg",
        subjects_dir=FREESURFER_SUBJECTS_DIR)

    if global_measures is None:
        msg = f"[WARN] Subject '{sid}': no global aseg measures extracted (ETIV/BSV normalization may not be possible for this subject)."
        print(msg)
        missing_global_measures_subjects.append(sid)
    else:
        for measure_name, measure_value in global_measures.items():
            colname = f"GLOBAL__{measure_name}"
            if colname not in df_pivot.columns:
                df_pivot[colname] = measure_value
            else:
                df_pivot[f"{colname}__dup"] = measure_value

    master_df_list.append(df_pivot)

# ---- Concatenate into master dataset
if len(master_df_list) == 0:
    raise RuntimeError("No subjects were successfully processed; master_df_list is empty.")

master_dataset = pd.concat(master_df_list, ignore_index=True)

print(f"\n[INFO] Built morphometry dataset: {master_dataset.shape[0]} rows × {master_dataset.shape[1]} columns")

if skipped_subjects:
    print(f"[INFO] Skipped subjects due to missing/failed stats files (n={len(skipped_subjects)}).")

if failed_subjects:
    print(f"[INFO] Subjects failed due to schema/other issues (n={len(failed_subjects)}).")

if missing_global_measures_subjects:
    print(f"[INFO] Subjects missing global aseg measures (n={len(missing_global_measures_subjects)}). Example: {missing_global_measures_subjects[:10]}")

# ---- Dataset-level QA (matches legacy spirit)
# Duplicate columns
column_names = master_dataset.columns.tolist()
column_name_counts = Counter(column_names)
duplicate_columns = {col: count for col, count in column_name_counts.items() if count > 1}
if duplicate_columns:
    print("\n[WARN] Duplicate column names found:")
    for col, count in duplicate_columns.items():
        print(f"  - '{col}' appears {count} times")
else:
    print("\n[INFO] No duplicate column names found.")

# Duplicate subject rows
duplicates = master_dataset[master_dataset.duplicated(subset=["subject_ID"], keep=False)]
if duplicates.empty:
    print("[INFO] No duplicate rows found (by subject_ID).")
else:
    print(f"[WARN] Duplicated subject_ID rows found: {duplicates['subject_ID'].nunique()} unique subject(s) duplicated.")

# Feature count (excluding IDs)
feature_columns = master_dataset.columns.difference(["subject_ID"])
print(f"[INFO] Number of features extracted: {len(feature_columns)}")

# Display head for sanity (avoid dumping huge table by default)
display(master_dataset.head())

In [ ]:
# __________________________________________________________________________________________________________
### FINALIZE DATASET FOR EXPORT + PRINT MISSINGNESS REPORT (no file I/O yet)

master_dataset_export = master_dataset.copy()

# ---- Enforce one-row-per-subject policy
if master_dataset_export["subject_ID"].duplicated().any():
    duplicate_subject_ids = (
        master_dataset_export.loc[master_dataset_export["subject_ID"].duplicated(), "subject_ID"]
        .unique()
        .tolist())
    msg = f"[WARN] Duplicate subject_ID rows detected (n={len(duplicate_subject_ids)}). Example: {duplicate_subject_ids[:10]}"
    if HARD_STOP:
        raise RuntimeError(msg)
    else:
        print(msg + "  --> Dropping duplicates, keeping first occurrence.")
        master_dataset_export = (
            master_dataset_export.drop_duplicates(subset=["subject_ID"], keep="first").reset_index(drop=True))

# ---- Split columns into: IDs, regional morphometry, and FreeSurfer global header measures
id_columns = ["subject_ID"]
global_measure_columns = sorted([column for column in master_dataset_export.columns if column.startswith("GLOBAL__")])
regional_feature_columns = sorted([
    column for column in master_dataset_export.columns
    if (column not in id_columns and not column.startswith("GLOBAL__"))])

# ---- Stabilize column order:
#      subject_ID first, then regional morphometry features, then GLOBAL__ measures
master_dataset_export = master_dataset_export[id_columns + regional_feature_columns + global_measure_columns]

# ---- Ensure numeric dtypes for all non-ID columns (coerce if needed)
all_feature_columns = regional_feature_columns + global_measure_columns
for column in all_feature_columns:
    if not pd.api.types.is_numeric_dtype(master_dataset_export[column]):
        master_dataset_export[column] = pd.to_numeric(master_dataset_export[column], errors="coerce")

# ---- Missingness summary (NaNs preserved)
n_subjects = master_dataset_export.shape[0]

n_regional_features = len(regional_feature_columns)
n_global_measures = len(global_measure_columns)
n_total_features = n_regional_features + n_global_measures

# Regional missingness (morphometry features only)
regional_na_matrix = master_dataset_export[regional_feature_columns].isna() if n_regional_features > 0 else None
regional_total_missing = int(regional_na_matrix.sum().sum()) if regional_na_matrix is not None else 0
regional_total_cells = int(n_subjects * n_regional_features)
regional_pct_missing = (regional_total_missing / regional_total_cells * 100.0) if regional_total_cells > 0 else 0.0

# Global-measure missingness
global_na_matrix = master_dataset_export[global_measure_columns].isna() if n_global_measures > 0 else None
global_total_missing = int(global_na_matrix.sum().sum()) if global_na_matrix is not None else 0
global_total_cells = int(n_subjects * n_global_measures)
global_pct_missing = (global_total_missing / global_total_cells * 100.0) if global_total_cells > 0 else 0.0

# Overall missingness (regional + global)
overall_total_missing = regional_total_missing + global_total_missing
overall_total_cells = regional_total_cells + global_total_cells
overall_pct_missing = (overall_total_missing / overall_total_cells * 100.0) if overall_total_cells > 0 else 0.0

# Missingness by subject (regional only, since this is what your legacy report conceptually tracked)
if regional_na_matrix is not None:
    regional_missing_by_subject = regional_na_matrix.sum(axis=1)  # number of missing regional features per subject
    subjects_with_regional_missing = master_dataset_export.loc[regional_missing_by_subject > 0, "subject_ID"].tolist()
else:
    regional_missing_by_subject = pd.Series([0] * n_subjects)
    subjects_with_regional_missing = []

# Missingness by feature (regional)
if regional_na_matrix is not None:
    regional_missing_by_feature = regional_na_matrix.sum(axis=0)  # number of missing subjects per regional feature
    regional_features_with_missing = regional_missing_by_feature[regional_missing_by_feature > 0].sort_values(ascending=False)
else:
    regional_missing_by_feature = pd.Series(dtype=int)
    regional_features_with_missing = pd.Series(dtype=int)

# Missingness by feature (global)
if global_na_matrix is not None:
    global_missing_by_feature = global_na_matrix.sum(axis=0)  # number of missing subjects per global measure
    global_features_with_missing = global_missing_by_feature[global_missing_by_feature > 0].sort_values(ascending=False)
else:
    global_missing_by_feature = pd.Series(dtype=int)
    global_features_with_missing = pd.Series(dtype=int)

print(f"[INFO] Export-ready table: {n_subjects} subjects × {n_regional_features} morphometry features (+ {n_global_measures} GLOBAL__ measures) = {n_total_features} total feature columns")
print(f"[INFO] Overall missingness (all features): {overall_total_missing}/{overall_total_cells} cells = {overall_pct_missing:.3f}%")

print(f"[INFO] Regional missingness (morphometry only): {regional_total_missing}/{regional_total_cells} cells = {regional_pct_missing:.3f}%")
print(f"[INFO] Global missingness (GLOBAL__ only): {global_total_missing}/{global_total_cells} cells = {global_pct_missing:.3f}%")

print(f"[INFO] Subjects with any missing REGIONAL features: {len(subjects_with_regional_missing)}/{n_subjects}")
print(f"[INFO] Regional features with any missing subjects: {int((regional_missing_by_feature > 0).sum()) if n_regional_features > 0 else 0}/{n_regional_features}")

if n_global_measures > 0:
    print(f"[INFO] Global measures with any missing subjects: {int((global_missing_by_feature > 0).sum())}/{n_global_measures}")

# ---- Explicit report (print + small displays)
# Adjust these caps if you want longer/shorter reports
MAX_SUBJECTS_TO_PRINT = 50
MAX_FEATURES_TO_PRINT = 50

if len(subjects_with_regional_missing) > 0:
    # Sort subjects by missing count descending (regional only)
    subject_report = (
        pd.DataFrame({
            "subject_ID": master_dataset_export["subject_ID"],
            "n_missing_regional_features": regional_missing_by_subject.astype(int)})
        .query("n_missing_regional_features > 0")
        .sort_values("n_missing_regional_features", ascending=False))

    print(f"\n[REPORT] Subjects missing ≥1 REGIONAL feature (showing up to {MAX_SUBJECTS_TO_PRINT}):")
    display(subject_report.head(MAX_SUBJECTS_TO_PRINT))

if len(regional_features_with_missing) > 0:
    regional_feature_report = regional_features_with_missing.astype(int).to_frame("n_missing_subjects")

    print(f"\n[REPORT] REGIONAL features missing in ≥1 subject (showing up to {MAX_FEATURES_TO_PRINT}):")
    display(regional_feature_report.head(MAX_FEATURES_TO_PRINT))

if len(global_features_with_missing) > 0:
    global_feature_report = global_features_with_missing.astype(int).to_frame("n_missing_subjects")

    print(f"\n[REPORT] GLOBAL__ measures missing in ≥1 subject (showing up to {MAX_FEATURES_TO_PRINT}):")
    display(global_feature_report.head(MAX_FEATURES_TO_PRINT))

# Print table summary:
display(master_dataset_export.head())

Post-processing: Combine corresponding LH+RH pairs (if enabled); standardize column names; etc.: 

In [ ]:
# __________________________________________________________________________________________________________
### FINAL EXPORT-PREP:
# Applies:
#   - Optional bilateral (Bilateral-*) feature creation (CALCULATE_BILATERAL_FEATURES)
#   - Optional semantic column standardization to Left-/Right-/Bilateral- (STANDARDIZE_COLUMN_NAMES)
#   - Universal prefix (MORPHOMETRY_COLUMN_PREFIX) applied to all feature column names
#
# Output:
#   - morphometry_export_df   (final, export-ready)
#
# Note: By default, NaN semantics are preserved: bilateral sums are NaN if either hemisphere is NaN.

# -----------------------------
# Naming control
# -----------------------------
BILATERAL_PREFIX = "Bilateral-"     # <-- Controls substring used to designate (synthetic) whole-brain measures

# Ensure prefix is a string (allow empty string to effectively disable):
MORPHOMETRY_COLUMN_PREFIX = "" if MORPHOMETRY_COLUMN_PREFIX is None else str(MORPHOMETRY_COLUMN_PREFIX)

df = master_dataset_export.copy()

# Identify column groups:
id_columns = ["subject_ID"]
global_feature_columns = [column for column in df.columns if column.startswith("GLOBAL__")]
regional_feature_columns = [column for column in df.columns if (column not in id_columns and not column.startswith("GLOBAL__"))]

# ---------------------------------------------------------------------
# A) Helper functions
# ---------------------------------------------------------------------

def _nan_preserving_sum(a: pd.Series, b: pd.Series) -> pd.Series:
    """NaN-preserving sum: result is NaN if either side is NaN."""
    return a + b

def _safe_add_column(df: pd.DataFrame, column_name: str, values: pd.Series) -> str:
    """
    Add a new column without overwriting existing columns.
    Returns the actual column name used (may be adjusted if collision).
    """
    if column_name not in df.columns:
        df[column_name] = values
        return column_name
    i = 2
    while f"{column_name}__{i}" in df.columns:
        i += 1
    new_name = f"{column_name}__{i}"
    df[new_name] = values
    return new_name

# ---------------------------------------------------------------------
# B) OPTIONAL: Standardize separators '_' -> '-'
#    IMPORTANT: Apply only to REGIONAL features (GLOBAL__ preserved exactly)
# ---------------------------------------------------------------------

if STANDARDIZE_COLUMN_NAMES:
    separator_rename_map = {
        column: column.replace("_", "-")
        for column in regional_feature_columns}
    df = df.rename(columns=separator_rename_map)

# Refresh column groups after optional rename
global_feature_columns = [column for column in df.columns if column.startswith("GLOBAL__")]
regional_feature_columns = [column for column in df.columns if (column not in id_columns and not column.startswith("GLOBAL__"))]

# ---------------------------------------------------------------------
# C) OPTIONAL: Semantic standardization of hemisphere coding
#    IMPORTANT: Apply only to REGIONAL features (GLOBAL__ preserved exactly)
# ---------------------------------------------------------------------

semantic_rename_map = {}
semantic_rule_hits = {"gm-lh": 0, "gm-rh": 0, "wm-lh": 0, "wm-rh": 0}

if STANDARDIZE_COLUMN_NAMES:
    for column in regional_feature_columns:
        new_column = column

        if column.startswith("gm-lh-"):
            new_column = "Left-gm-" + column[len("gm-lh-"):]
            semantic_rule_hits["gm-lh"] += 1
        elif column.startswith("gm-rh-"):
            new_column = "Right-gm-" + column[len("gm-rh-"):]
            semantic_rule_hits["gm-rh"] += 1
        elif column.startswith("wm-lh-"):
            new_column = "Left-wm-" + column[len("wm-lh-"):]
            semantic_rule_hits["wm-lh"] += 1
        elif column.startswith("wm-rh-"):
            new_column = "Right-wm-" + column[len("wm-rh-"):]
            semantic_rule_hits["wm-rh"] += 1

        if new_column != column:
            if new_column in df.columns:
                i = 2
                candidate = f"{new_column}__{i}"
                while candidate in df.columns:
                    i += 1
                    candidate = f"{new_column}__{i}"
                new_column = candidate
            semantic_rename_map[column] = new_column

    df = df.rename(columns=semantic_rename_map)

# Refresh column groups after semantic rename
global_feature_columns = [column for column in df.columns if column.startswith("GLOBAL__")]
regional_feature_columns = [column for column in df.columns if (column not in id_columns and not column.startswith("GLOBAL__"))]

# ---------------------------------------------------------------------
# D) OPTIONAL: Create bilateral (Bilateral-*) features
#    IMPORTANT: Operate only on REGIONAL features
# ---------------------------------------------------------------------

bilateral_created = []  # (new_col, left_col, right_col, family)

if CALCULATE_BILATERAL_FEATURES:

    regional_feature_columns_set = set(regional_feature_columns)

    # Family 1: Left-/Right-
    left_columns = [column for column in regional_feature_columns if column.startswith("Left-")]
    for left_column in left_columns:
        base = left_column[len("Left-"):]
        right_column = "Right-" + base
        if right_column in regional_feature_columns_set:
            new_column = BILATERAL_PREFIX + base
            used_name = _safe_add_column(df, new_column, _nan_preserving_sum(df[left_column], df[right_column]))
            bilateral_created.append((used_name, left_column, right_column, "LH + RH"))

    # Raw gm / wm patterns (only if not standardized)
    if not STANDARDIZE_COLUMN_NAMES:
        gm_lh_columns = [
            column for column in regional_feature_columns
            if column.startswith("gm_lh_") or column.startswith("gm-lh-")]
        gm_rh_set = {
            column for column in regional_feature_columns
            if column.startswith("gm_rh_") or column.startswith("gm-rh-")}

        for left_column in gm_lh_columns:
            if left_column.startswith("gm_lh_"):
                base = left_column[len("gm_lh_"):]
                right_column = "gm_rh_" + base
            else:
                base = left_column[len("gm-lh-"):]
                right_column = "gm-rh-" + base

            if right_column in gm_rh_set:
                new_column = BILATERAL_PREFIX + "gm-" + base.replace("_", "-")
                used_name = _safe_add_column(
                    df,
                    new_column,
                    _nan_preserving_sum(df[left_column], df[right_column]))
                bilateral_created.append((used_name, left_column, right_column, "gm_lh_rh_raw"))

        wm_lh_columns = [column for column in regional_feature_columns if column.startswith("wm-lh-")]
        wm_rh_set = {column for column in regional_feature_columns if column.startswith("wm-rh-")}

        for left_column in wm_lh_columns:
            base = left_column[len("wm-lh-"):]
            right_column = "wm-rh-" + base
            if right_column in wm_rh_set:
                new_column = BILATERAL_PREFIX + "wm-" + base
                used_name = _safe_add_column(
                    df,
                    new_column,
                    _nan_preserving_sum(df[left_column], df[right_column]))
                bilateral_created.append((used_name, left_column, right_column, "wm_lh_rh_raw"))

# ---------------------------------------------------------------------
# E) Apply universal morphometry prefix
#    IMPORTANT: Apply only to REGIONAL features (GLOBAL__ preserved exactly)
# ---------------------------------------------------------------------

# Recompute lists after bilateral creation
global_feature_columns = [column for column in df.columns if column.startswith("GLOBAL__")]
regional_feature_columns = [column for column in df.columns if (column not in id_columns and not column.startswith("GLOBAL__"))]

if MORPHOMETRY_COLUMN_PREFIX:
    prefix_rename_map = {
        column: MORPHOMETRY_COLUMN_PREFIX + column
        for column in regional_feature_columns}
    df = df.rename(columns=prefix_rename_map)

# ---------------------------------------------------------------------
# F) Final ordering
#    subject_ID first, then prefixed regional features (sorted), then GLOBAL__ (sorted)
# ---------------------------------------------------------------------

global_feature_columns = sorted([column for column in df.columns if column.startswith("GLOBAL__")])
regional_feature_columns = sorted([column for column in df.columns if (column not in id_columns and not column.startswith("GLOBAL__"))])

morphometry_export_df = df[["subject_ID"] + regional_feature_columns + global_feature_columns]

# ---------------------------------------------------------------------
# G) Reporting
# ---------------------------------------------------------------------

print("\n[INFO] Morphometry export-prep complete.")
print(f"[INFO] STANDARDIZE_COLUMN_NAMES: {STANDARDIZE_COLUMN_NAMES}")
print(f"[INFO] CALCULATE_BILATERAL_FEATURES: {CALCULATE_BILATERAL_FEATURES}")
print(f"[INFO] Bilateral label: '{BILATERAL_PREFIX}'")
print(f"[INFO] Prefix applied (REGIONAL only): '{MORPHOMETRY_COLUMN_PREFIX}'" if MORPHOMETRY_COLUMN_PREFIX else "[INFO] Prefix applied (REGIONAL only): <none>")
print(f"[INFO] GLOBAL__ columns preserved (unmodified): {len(global_feature_columns)}")

if STANDARDIZE_COLUMN_NAMES:
    n_semantic_renamed = sum(semantic_rule_hits.values())
    print(f"[INFO] Semantic hemisphere standardization applied to {n_semantic_renamed} REGIONAL feature columns:")
    for rule_name, rule_count in semantic_rule_hits.items():
        print(f"       - {rule_name}: {rule_count}")
else:
    print("[INFO] Semantic hemisphere standardization: disabled (original REGIONAL FreeSurfer names retained)")

if CALCULATE_BILATERAL_FEATURES:
    print(f"[INFO] Bilateral REGIONAL features created: {len(bilateral_created)}")
    if bilateral_created:
        created_df = pd.DataFrame(
            bilateral_created,
            columns=["new_feature", "left_feature", "right_feature", "family"])
        print("\n[REPORT] Bilateral feature families (counts):")
        display(created_df["family"].value_counts().to_frame("n_created"))
        print("\n[REPORT] Examples of Bilateral features:")
        display(created_df.head(10))

print(f"\n[INFO] Final export-ready table: {morphometry_export_df.shape[0]} rows × {morphometry_export_df.shape[1]} columns")
display(morphometry_export_df.sample(min(morphometry_export_df.shape[0],10)))

--------

### Normalization (via eTIV and/or BSV):

- (Technically optional, but highly recommended for methodological robustness)

In [ ]:
# __________________________________________________________________________________________________________
### (OPTIONAL) NORMALIZATION:
# - Relabel existing morphometry_export_df regional features as RAW: <BASE_PREFIX><feature> -> <BASE_PREFIX>RAW__<feature>
# - Optionally add ETIV- and/or BSV-normalized feature blocks
# - Supports EXPORT_MODE: "single" or "separate"
#
# Outputs:
# - morphometry_export_single_df  (if EXPORT_MODE == "single")
# - morphometry_export_tables     (dict of {rep: df} if EXPORT_MODE == "separate")


# -----------------------------
# A) Helpers: normalization of tokens
# -----------------------------
def _norm_token(value) -> str:
    return str(value).strip().lower().replace("-", "").replace("_", "")

def _normalize_export_mode(export_mode_value: str) -> str:
    mode = _norm_token(export_mode_value)
    if mode not in {"single", "separate"}:
        raise ValueError(f"Unrecognized EXPORT_MODE: '{export_mode_value}' (allowed: single, separate)")
    return mode

def _normalize_representations(export_representations_value) -> list:
    if isinstance(export_representations_value, str):
        representation_tokens = [token.strip() for token in export_representations_value.split(",") if token.strip()]
    else:
        representation_tokens = list(export_representations_value)

    normalized = []
    for token in representation_tokens:
        normalized_token = _norm_token(token)
        if normalized_token in ("raw",):
            normalized.append("raw")
        elif normalized_token in ("etiv", "etivnorm"):
            normalized.append("etiv")
        elif normalized_token in ("bsv", "brainseg", "brainsegvol", "brainsegvolnotvent"):
            normalized.append("bsv")
        else:
            raise ValueError(f"Unrecognized export representation token: '{token}' (allowed: RAW, ETIV, BSV)")

    if "raw" not in normalized:
        normalized = ["raw"] + normalized

    normalized = list(dict.fromkeys(normalized))
    return normalized


# -----------------------------
# B) Parse mode/representations from config vars
# -----------------------------
normalized_representations = _normalize_representations(EXPORT_REPRESENTATIONS)
export_mode_normalized = _normalize_export_mode(EXPORT_MODE)

print(f"[INFO] EXPORT_REPRESENTATIONS (normalized): {normalized_representations}")
print(f"[INFO] EXPORT_MODE (normalized): {export_mode_normalized}")


# -----------------------------
# C) Establish BASE_PREFIX and start from morphometry_export_df
# -----------------------------
BASE_PREFIX = "" if MORPHOMETRY_COLUMN_PREFIX is None else str(MORPHOMETRY_COLUMN_PREFIX)

input_dataframe = morphometry_export_df.copy()
if "subject_ID" not in input_dataframe.columns:
    raise KeyError("Expected 'subject_ID' column not found in morphometry_export_df.")


# -----------------------------
# D) Filter GLOBAL__ columns to a user-specified keep-list
# -----------------------------
GLOBAL_KEEP_LIST = [
    "GLOBAL__eTIV",
    "GLOBAL__BrainSegVolNotVent",
    "GLOBAL__CortexVol",
    "GLOBAL__CerebralWhiteMatterVol",
    "GLOBAL__SubCortGrayVol",
    "GLOBAL__TotalGrayVol",
    "GLOBAL__VentricleChoroidVol"]

available_global_columns = [column for column in input_dataframe.columns if column.startswith("GLOBAL__")]
global_keep_columns = [column for column in GLOBAL_KEEP_LIST if column in input_dataframe.columns]
global_missing_keep_columns = [column for column in GLOBAL_KEEP_LIST if column not in input_dataframe.columns]

if global_missing_keep_columns:
    print(f"[WARN] Requested GLOBAL__ columns not found and will be omitted: {global_missing_keep_columns}")

global_drop_columns = [column for column in available_global_columns if column not in global_keep_columns]
if global_drop_columns:
    input_dataframe = input_dataframe.drop(columns=global_drop_columns)

global_columns = global_keep_columns


# -----------------------------
# E) Relabel REGIONAL columns as RAW__ (do not touch GLOBAL__)
# -----------------------------
regional_columns = [
    column for column in input_dataframe.columns
    if (column != "subject_ID" and not column.startswith("GLOBAL__"))]

raw_rename_map = {}
for column in regional_columns:
    if BASE_PREFIX and column.startswith(BASE_PREFIX):
        suffix = column[len(BASE_PREFIX):]
        new_column = f"{BASE_PREFIX}RAW__{suffix}"
    else:
        new_column = f"{BASE_PREFIX}RAW__{column}" if BASE_PREFIX else f"RAW__{column}"
    raw_rename_map[column] = new_column

df_raw = input_dataframe.rename(columns=raw_rename_map)


# -----------------------------
# F) Denominator discovery (must be GLOBAL__ columns)
# -----------------------------
ETIV_CANDIDATES = ["EstimatedTotalIntraCranialVol", "eTIV", "ETIV"]
BSV_CANDIDATES = ["BrainSegVolNotVent", "BrainSegVol", "BrainSegVolNotVentSurf"]

def _find_global_denominator_column(dataframe: pd.DataFrame, candidates: list) -> Optional[str]:
    for candidate in candidates:
        candidate_str = str(candidate)
        candidate_column = f"GLOBAL__{candidate_str}"
        if candidate_column in dataframe.columns:
            return candidate_column
    return None

etiv_global_column = _find_global_denominator_column(df_raw, ETIV_CANDIDATES)
bsv_global_column = _find_global_denominator_column(df_raw, BSV_CANDIDATES)

print(f"[INFO] Detected eTIV denominator GLOBAL column: {etiv_global_column if etiv_global_column else '<not found>'}")
print(f"[INFO] Detected BSV denominator GLOBAL column:  {bsv_global_column if bsv_global_column else '<not found>'}")


# -----------------------------
# G) Vectorized normalized block creation
# -----------------------------
def _make_normalized_block_vectorized(
    df_raw: pd.DataFrame,
    base_prefix: str,
    denom_global_column: str,
    representation_tag: str
) -> pd.DataFrame:
    raw_tag = f"{base_prefix}RAW__" if base_prefix else "RAW__"
    representation_prefix = f"{base_prefix}{representation_tag.upper()}__" if base_prefix else f"{representation_tag.upper()}__"

    raw_regional_columns = [column for column in df_raw.columns if column.startswith(raw_tag)]
    if len(raw_regional_columns) == 0:
        return df_raw[["subject_ID"]].copy()

    numerator_matrix = df_raw[raw_regional_columns].astype(float)
    denominator_series = df_raw[denom_global_column].astype(float).replace(0.0, np.nan)

    normalized_matrix = numerator_matrix.div(denominator_series, axis=0)

    rename_map = {}
    for raw_column in raw_regional_columns:
        suffix = raw_column[len(raw_tag):]
        rename_map[raw_column] = representation_prefix + suffix

    normalized_matrix = normalized_matrix.rename(columns=rename_map)
    output_block = pd.concat([df_raw[["subject_ID"]], normalized_matrix], axis=1).copy()
    return output_block


# -----------------------------
# H) Build blocks dict (regional-only; globals added later exactly once)
# -----------------------------
raw_tag = f"{BASE_PREFIX}RAW__" if BASE_PREFIX else "RAW__"
raw_regional_columns_sorted = sorted(
    [column for column in df_raw.columns if (column != "subject_ID" and column.startswith(raw_tag))])

blocks = {"raw": df_raw[["subject_ID"] + raw_regional_columns_sorted].copy()}

if "etiv" in normalized_representations:
    if etiv_global_column is None:
        msg = "[WARN] ETIV normalization requested, but no GLOBAL__ eTIV denominator column was found."
        if HARD_STOP:
            raise RuntimeError(msg)
        else:
            print(msg + "  --> Skipping ETIV block.")
    else:
        df_etiv = _make_normalized_block_vectorized(df_raw, BASE_PREFIX, etiv_global_column, representation_tag="ETIV")
        blocks["etiv"] = df_etiv
        print(f"[INFO] ETIV block created with {df_etiv.shape[1]-1} features.")

if "bsv" in normalized_representations:
    if bsv_global_column is None:
        msg = "[WARN] BSV normalization requested, but no GLOBAL__ BrainSegVol* denominator column was found."
        if HARD_STOP:
            raise RuntimeError(msg)
        else:
            print(msg + "  --> Skipping BSV block.")
    else:
        df_bsv = _make_normalized_block_vectorized(df_raw, BASE_PREFIX, bsv_global_column, representation_tag="BSV")
        blocks["bsv"] = df_bsv
        print(f"[INFO] BSV block created with {df_bsv.shape[1]-1} features.")


# -----------------------------
# I) Final global prefixing: GLOBAL__X -> <BASE_PREFIX>GLOBAL__X
# -----------------------------
def _apply_global_prefixing(dataframe: pd.DataFrame, base_prefix: str) -> pd.DataFrame:
    rename_map = {}
    for column in dataframe.columns:
        if column.startswith("GLOBAL__"):
            rename_map[column] = f"{base_prefix}GLOBAL__{column[len('GLOBAL__'):]}"
    if rename_map:
        dataframe = dataframe.rename(columns=rename_map)
    return dataframe


# -----------------------------
# J) Assemble outputs (single vs separate) using EXPORT_MODE
# -----------------------------
if export_mode_normalized == "single":
    morphometry_export_single_df = blocks["raw"].copy()

    for representation_key, representation_dataframe in blocks.items():
        if representation_key == "raw":
            continue
        morphometry_export_single_df = morphometry_export_single_df.merge(
            representation_dataframe, on="subject_ID", how="left")

    # Add GLOBAL columns exactly once, already prefixed into morphometry namespace
    if global_columns:
        global_dataframe = df_raw[["subject_ID"] + global_columns].copy()
        global_dataframe = _apply_global_prefixing(global_dataframe, BASE_PREFIX)
        morphometry_export_single_df = morphometry_export_single_df.merge(
            global_dataframe, on="subject_ID", how="left")

    final_columns = ["subject_ID"] + sorted(
        [column for column in morphometry_export_single_df.columns if column != "subject_ID"])
    morphometry_export_single_df = morphometry_export_single_df[final_columns].copy()

    print(f"\n[INFO] Single-table export assembled: {morphometry_export_single_df.shape[0]} rows × {morphometry_export_single_df.shape[1]} columns")
    display(morphometry_export_single_df.head())

else:
    morphometry_export_tables = {}
    for representation_key, representation_dataframe in blocks.items():
        representation_with_globals = representation_dataframe.copy()

        if global_columns:
            global_dataframe = df_raw[["subject_ID"] + global_columns].copy()
            global_dataframe = _apply_global_prefixing(global_dataframe, BASE_PREFIX)
            representation_with_globals = representation_with_globals.merge(
                global_dataframe, on="subject_ID", how="left")

        final_columns = ["subject_ID"] + sorted(
            [column for column in representation_with_globals.columns if column != "subject_ID"])
        morphometry_export_tables[representation_key] = representation_with_globals[final_columns].copy()

    print(f"\n[INFO] Separate-table export prepared for representations: {list(morphometry_export_tables.keys())}")
    for representation_key, representation_dataframe in morphometry_export_tables.items():
        print(f"       - {representation_key.upper()}: {representation_dataframe.shape[0]} rows × {representation_dataframe.shape[1]} cols")

-------

### Final save / export:

In [ ]:
# __________________________________________________________________________________________________________
### FINAL SAVE / EXPORT (CSV)

def _norm_token(value) -> str:
    return str(value).strip().lower().replace("-", "").replace("_", "")

def _normalize_export_mode(export_mode_value: str) -> str:
    mode = _norm_token(export_mode_value)
    if mode not in {"single", "separate"}:
        raise ValueError(f"Unrecognized EXPORT_MODE: '{export_mode_value}' (allowed: single, separate)")
    return mode

def _normalize_representations(export_representations_value) -> list:
    if isinstance(export_representations_value, str):
        representation_tokens = [token.strip() for token in export_representations_value.split(",") if token.strip()]
    else:
        representation_tokens = list(export_representations_value)

    normalized = []
    for token in representation_tokens:
        normalized_token = _norm_token(token)
        if normalized_token in ("raw",):
            normalized.append("raw")
        elif normalized_token in ("etiv", "etivnorm"):
            normalized.append("etiv")
        elif normalized_token in ("bsv", "brainseg", "brainsegvol", "brainsegvolnotvent"):
            normalized.append("bsv")
        else:
            raise ValueError(f"Unrecognized export representation token: '{token}' (allowed: RAW, ETIV, BSV)")

    # RAW always exists conceptually; ensure included
    if "raw" not in normalized:
        normalized = ["raw"] + normalized

    # de-duplicate preserve order
    normalized = list(dict.fromkeys(normalized))
    return normalized

# Normalize mode + reps (do not assume earlier cells left normalized vars around)
export_mode_normalized = _normalize_export_mode(EXPORT_MODE)
representations_normalized = _normalize_representations(EXPORT_REPRESENTATIONS)

# OUTPUT_PATH is defined in init; verify
if "OUTPUT_PATH" not in globals():
    raise RuntimeError("OUTPUT_PATH is not defined. Ensure your init cell created OUTPUT_PATH.")

OUTPUT_PATH = Path(OUTPUT_PATH)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Timestamp for reproducibility (optional; can remove if you prefer stable filenames)
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Determine a stable base prefix for filenames
file_prefix = "morphometry"

# Write
written_files = []

if export_mode_normalized == "single":
    if "morphometry_export_single_df" not in globals():
        raise RuntimeError("morphometry_export_single_df not found. Run the normalization cell first (single mode).")

    output_file = OUTPUT_PATH / f"{file_prefix}.csv"
    morphometry_export_single_df.to_csv(output_file, index=False)
    written_files.append(output_file)

    print(f"[INFO] Wrote SINGLE export: {output_file}")
    print(f"[INFO] Shape: {morphometry_export_single_df.shape[0]} rows × {morphometry_export_single_df.shape[1]} columns")

else:
    if "morphometry_export_tables" not in globals():
        raise RuntimeError("morphometry_export_tables not found. Run the normalization cell first (separate mode).")

    # Only write the representations requested (and actually produced)
    available_tables = set(morphometry_export_tables.keys())
    requested_tables = [rep for rep in representations_normalized if rep in available_tables]

    if len(requested_tables) == 0:
        raise RuntimeError(
            f"No requested representations found in morphometry_export_tables. "
            f"Requested: {representations_normalized}; Available: {sorted(list(available_tables))}")

    for rep in requested_tables:
        output_file = OUTPUT_PATH / f"{file_prefix}_{rep.upper()}.csv"
        morphometry_export_tables[rep].to_csv(output_file, index=False)
        written_files.append(output_file)
        print(f"[INFO] Wrote {rep.upper()} export: {output_file}  |  Shape: {morphometry_export_tables[rep].shape}")

print(f"\n[INFO] Export complete. Files written: {len(written_files)}")
for path in written_files:
    print(f"  - {path}")

--------